# 01 — Dataset EDA
Explore the ArXiv snapshot and CiteULike interactions loaded into Postgres.

In [ ]:
import os
os.chdir('..')  # repo root
from dotenv import load_dotenv
load_dotenv()

import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. ArXiv snapshot stats

In [ ]:
arxiv_path = Path(os.environ['ARXIV_DATA_PATH'])
records = []
with arxiv_path.open() as f:
    for i, line in enumerate(f):
        if i >= 200_000:
            break
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

df = pd.DataFrame(records)
print(f'Loaded {len(df):,} records (first 200k)')
df.head(3)

In [ ]:
# Papers per year
df['year'] = df['id'].str[:2].astype(int).apply(lambda y: 2000+y if y < 90 else 1900+y)
year_counts = df['year'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(year_counts.index, year_counts.values, color='steelblue')
ax.set_title('ArXiv papers per year (first 200k)')
ax.set_xlabel('Year'); ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
plt.tight_layout(); plt.show()

In [ ]:
# Top categories
cats = Counter()
for row in df['categories'].dropna():
    for c in str(row).split():
        cats[c] += 1

top = pd.Series(dict(cats.most_common(20)))
fig, ax = plt.subplots(figsize=(10, 5))
top.sort_values().plot.barh(ax=ax, color='teal')
ax.set_title('Top 20 ArXiv categories'); ax.set_xlabel('Paper count')
plt.tight_layout(); plt.show()

In [ ]:
# Abstract length distribution
df['abs_len'] = df['abstract'].dropna().str.split().apply(len)
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['abs_len'].dropna(), bins=60, color='darkorange', edgecolor='white')
ax.set_title('Abstract length (words)'); ax.set_xlabel('Words'); ax.set_ylabel('Papers')
plt.tight_layout(); plt.show()
print(df['abs_len'].describe())

## 2. DB stats (requires running Postgres)

In [ ]:
import asyncio
from sqlalchemy.ext.asyncio import create_async_engine
from sqlalchemy import text

engine = create_async_engine(os.environ['DATABASE_URL'])

async def db_stats():
    async with engine.connect() as conn:
        r = await conn.execute(text('SELECT COUNT(*) FROM papers'))
        n_papers = r.scalar()
        r2 = await conn.execute(text('SELECT COUNT(DISTINCT user_id) FROM user_history'))
        n_users = r2.scalar()
        r3 = await conn.execute(text('SELECT COUNT(*) FROM user_history'))
        n_interactions = r3.scalar()
    print(f'Papers in DB : {n_papers:,}')
    print(f'Users        : {n_users:,}')
    print(f'Interactions : {n_interactions:,}')

await db_stats()

In [ ]:
# Interactions per user (sparsity)
async def interactions_per_user():
    async with engine.connect() as conn:
        r = await conn.execute(text(
            "SELECT user_id, COUNT(*) as cnt FROM user_history GROUP BY user_id"
        ))
        rows = r.fetchall()
    counts = [row[1] for row in rows]
    s = pd.Series(counts)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(s, bins=50, log=True, color='purple', edgecolor='white')
    ax.set_title('Interactions per user (log scale)')
    ax.set_xlabel('# papers saved'); ax.set_ylabel('# users')
    plt.tight_layout(); plt.show()
    print(s.describe())

await interactions_per_user()